In [1]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

In [2]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
from datetime import timedelta, date
import itertools, pandas
#warnings.filterwarnings('ignore')

In [3]:
a = pd.read_excel('./To Shivam C.xlsx')

In [4]:
df = a.transpose()
new_header = df.iloc[0]
df = df[1:]
df.columns = new_header
df.head()

Unnamed: 0,Customer Acquisition Assumptions,No of Registrations,Android,iOS,nan,nan,Active App User Assumptions,Android,D30,D60,...,D450,D480,D510,D540,D570,D600,D630,D660,D690,D720
(Jul'19-Sep'19),NaN,NaN,962500,137500,NaN,NaN,NaN,NaN,0.1,0.08,...,0.01,0.005,0.005,0.005,0.005,0.005,0.005,0.005,0.005,0.005
(Oct'19-Dec'19),NaN,NaN,1.4875e+06,212500,NaN,NaN,NaN,NaN,0.11,0.09,...,0.02,0.015,0.015,0.015,0.015,0.015,0.015,0.015,0.015,0.015
(Jan'20-Mar'20),NaN,NaN,1.8375e+06,262500,NaN,NaN,NaN,NaN,0.11,0.09,...,0.02,0.015,0.015,0.015,0.015,0.015,0.015,0.015,0.015,0.015
(Apr'20-Jun'20),NaN,NaN,3.5e+06,500000,NaN,NaN,NaN,NaN,0.11,0.09,...,0.02,0.015,0.015,0.015,0.015,0.015,0.015,0.015,0.015,0.015
(Jul'20-Sep'20),NaN,NaN,3.675e+06,525000,NaN,NaN,NaN,NaN,0.11,0.09,...,0.02,0.015,0.015,0.015,0.015,0.015,0.015,0.015,0.015,0.015


In [5]:
dlist = []
for i in range(24):
    d_previous = (a.iloc[(8+i),1:].tolist())
    d_previous = [item for item in d_previous for j in range(3)]
    dlist.append(d_previous)

In [6]:
df_android = df[['Android']]
df_android.columns = ['android','nan']
df_android.drop('nan',inplace=True,axis =1)

/home/aurora/miniconda3/lib/python3.7/site-packages/pandas/core/frame.py:3697: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  errors=errors)


In [7]:
split = df_android['android'].tolist()
split[:] = [x // 3 for x in split]
split = [item for item in split for i in range(3)]
df_android_month = pd.DataFrame({'android':split})

In [8]:
def daterange(date1, date2):
    for n in range(int ((date2 - date1).days)+1):
        yield date1 + timedelta(n)
date_list = []
start_dt = date(19,7,1)
end_dt = date(21,12,31)
for dt in daterange(start_dt, end_dt):
    date_list.append(dt.strftime("%Y-%m"))
    
xy = pd.DataFrame({'months': date_list})
xy.drop_duplicates('months',inplace = True)
xy.reset_index(drop=True, inplace=True)

In [9]:
z = pd.merge(xy, df_android_month , left_index=True,right_index = True)
z.set_index('months',inplace=True)

In [10]:
zero = []
for i in range(24):
    temp = []
    for j in range(i+1):
        temp.append(0)
    zero.append(temp)

In [11]:
anusers = z['android'].tolist()
columns = []
for i in range(24):
    temp1 = zero[i] + anusers
    temp1 = temp1[:30]
    columns.append(temp1)

In [12]:
dlist1 = []
for i in range(24):
    temp1 = zero[i] + dlist[i]
    temp1 = temp1[:30]
    dlist1.append(temp1)

In [13]:
c1 = pd.DataFrame((_ for _ in itertools.zip_longest(*columns)), columns=['c1', 'c2', 'c3','c4','c5','c6','c7','c8','c9',
                                                                         'c10','c11','c12','c13','c14','c15','c16','c17',
                                                                         'c18','c19','c20','c21','c22','c23','c24'])
c2 = pd.DataFrame((_ for _ in itertools.zip_longest(*dlist1)), columns=['c1', 'c2', 'c3','c4','c5','c6','c7','c8','c9',
                                                                        'c10','c11','c12','c13','c14','c15','c16','c17',
                                                                        'c18','c19','c20','c21','c22','c23','c24'])

In [14]:
c3 = pd.DataFrame(c1.values*c2.values, columns=c1.columns, index=c1.index)

In [15]:
c4 = c3.sum(axis=1)
c4 = c4.to_frame().reset_index()
c4.columns = ['index','totals']
c4 = c4[['totals']]
c4 = c4.set_index(z.index)

In [16]:
df_android_final = pd.merge(z, c4, left_index=True, right_index=True)

In [17]:
df_android_final['total_android'] = df_android_final.sum(axis=1)

In [18]:
df_android_final

,android,totals,total_android
months,,,
19-07,320833.0,0.000,320833.000
19-08,320833.0,32083.300,352916.300
19-09,320833.0,57749.940,378582.940
19-10,495833.0,73791.590,569624.590
19-11,495833.0,109083.240,604916.240
19-12,495833.0,137666.560,633499.560
20-01,612500.0,159395.715,771895.715
20-02,612500.0,192208.240,804708.240
20-03,612500.0,220937.425,833437.425
